# Day 8: Data Preprocessing & Data Cleaning

**Intern:** Shri Sanjaykumar V  
**Role:** AI/ML Intern  
**Organization:** Linkific  
**Date:** 07 September 2026  

---

## 🎯 Learning Objectives & Resources

### Learning Objectives:
- **Understand why data preprocessing is important:** Raw real-world data is rarely clean; it contains missing entries, redundant duplicates, unstandardized naming, and improper data types that distort analysis and break machine learning models.
- **Learn basic data cleaning techniques:** Practical, reproducible data transformation techniques using Pandas.

### 📺 Recommended Learning Resources:
- **YouTube Search Topics:**
  - Data Cleaning in Python
  - Data Preprocessing using Pandas
  - Missing Values in Machine Learning
- **Recommended Channels:**
  - Krish Naik
  - CampusX
  - Codebasics
- **Documentation:**
  - [Pandas Documentation – Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)

### 💻 Tasks Covered in this Notebook:
1. **Task 1:** Load a dataset of your choice.
2. **Task 2:** Identify missing values.
3. **Task 3:** Handle missing values using appropriate techniques.
4. **Task 4:** Remove duplicate records.
5. **Task 5:** Rename columns where necessary.
6. **Task 6:** Convert incorrect data types.
7. **Task 7:** Save the cleaned dataset.
8. **Two-CSV Comparison:** Direct comparison between `dataset.csv` (raw) and `cleaned_dataset.csv` (cleaned).

---

## Library Imports

We import **Pandas** for data manipulation and tabular transformations, **NumPy** for numerical computations, and **os** for filesystem path resolution.

In [1]:
import os
import pandas as pd
import numpy as np

print("Pandas Version :", pd.__version__)
print("NumPy Version  :", np.__version__)
print("Environment initialized successfully.")

Pandas Version : 3.0.3
NumPy Version  : 2.5.0
Environment initialized successfully.


## 💻 Task 1: Load a Dataset of Your Choice

We load `dataset.csv`, which contains employee records from the **City of Houston Public Payroll Dataset**.
We inspect the dataset dimensions, first few rows, and column properties.

In [2]:
# Auto-detect dataset path whether executing from Day-8 folder or repository root
possible_paths = [
    "dataset.csv",
    os.path.join("Day-8", "dataset.csv"),
    os.path.join("Python", "Day-8", "dataset.csv")
]
data_path = next((p for p in possible_paths if os.path.exists(p)), "dataset.csv")
df = pd.read_csv(data_path)
print(f"Dataset loaded from: {os.path.abspath(data_path)}")
print(f"Raw Dataset Shape : {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded from: C:\Users\Priya\Downloads\internship\Day-8\dataset.csv
Raw Dataset Shape : 2005 rows, 10 columns


In [3]:
# Display first 5 records of the raw dataset
df.head()

Out[0]: 
   UNIQUE_ID               POSITION_TITLE  ...   HIRE_DATE    JOB_DATE
0          0  ASSISTANT DIRECTOR (EX LVL)  ...  2006-06-12  2012-10-13
1          1            LIBRARY ASSISTANT  ...  2000-07-19  2010-09-18
2          2               POLICE OFFICER  ...  2015-02-03  2015-02-03
3          3            ENGINEER/OPERATOR  ...  1982-02-08  1991-05-25
4          4                  ELECTRICIAN  ...  1989-06-19  1994-10-22

[5 rows x 10 columns]


,UNIQUE_ID,POSITION_TITLE,DEPARTMENT,BASE_SALARY,RACE,EMPLOYMENT_TYPE,GENDER,EMPLOYMENT_STATUS,HIRE_DATE,JOB_DATE
0,0,ASSISTANT DIRECTOR (EX LVL),Municipal Courts Department,121862.0,Hispanic/Latino,Full Time,Female,Active,2006-06-12,2012-10-13
1,1,LIBRARY ASSISTANT,Library,26125.0,Hispanic/Latino,Full Time,Female,Active,2000-07-19,2010-09-18
2,2,POLICE OFFICER,Houston Police Department-HPD,45279.0,White,Full Time,Male,Active,2015-02-03,2015-02-03
3,3,ENGINEER/OPERATOR,Houston Fire Department (HFD),63166.0,White,Full Time,Male,Active,1982-02-08,1991-05-25
4,4,ELECTRICIAN,General Services Department,56347.0,White,Full Time,Male,Active,1989-06-19,1994-10-22


In [4]:
# Inspect column data types and structural info
print("--- Column Data Types ---")
print(df.dtypes)
print("\n--- DataFrame Info ---")
df.info()

--- Column Data Types ---
UNIQUE_ID              int64
POSITION_TITLE           str
DEPARTMENT               str
BASE_SALARY          float64
RACE                     str
EMPLOYMENT_TYPE          str
GENDER                   str
EMPLOYMENT_STATUS        str
HIRE_DATE                str
JOB_DATE                 str
dtype: object

--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 2005 entries, 0 to 2004
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   UNIQUE_ID          2005 non-null   int64  
 1   POSITION_TITLE     2005 non-null   str    
 2   DEPARTMENT         2005 non-null   str    
 3   BASE_SALARY        1891 non-null   float64
 4   RACE               1970 non-null   str    
 5   EMPLOYMENT_TYPE    2005 non-null   str    
 6   GENDER             2005 non-null   str    
 7   EMPLOYMENT_STATUS  2005 non-null   str    
 8   HIRE_DATE          2005 non-null   str    
 9   JOB_DATE         

## 💻 Task 2: Identify Missing Values

Missing values (`NaN` / `null`) can skew statistical metrics and cause errors in analytical algorithms.
We identify missing values across all features using `df.isnull().sum()`.

In [5]:
print("--- Missing Values Count Per Column ---")
missing_counts = df.isnull().sum()
print(missing_counts)

total_missing = missing_counts.sum()
print(f"\nTotal Missing Values Across Dataset: {total_missing}")
print("Columns with Missing Values:")
print(dict(missing_counts[missing_counts > 0]))

--- Missing Values Count Per Column ---
UNIQUE_ID              0
POSITION_TITLE         0
DEPARTMENT             0
BASE_SALARY          114
RACE                  35
EMPLOYMENT_TYPE        0
GENDER                 0
EMPLOYMENT_STATUS      0
HIRE_DATE              0
JOB_DATE               3
dtype: int64

Total Missing Values Across Dataset: 152
Columns with Missing Values:
{'BASE_SALARY': np.int64(114), 'RACE': np.int64(35), 'JOB_DATE': np.int64(3)}


## 💻 Task 3: Handle Missing Values Using Appropriate Techniques

Rather than dropping rows and losing data, we apply context-specific imputation techniques:
1. **`BASE_SALARY` (114 missing)**: Numerical salary distributions are right-skewed by high-earning management. The **median** is outlier-robust and preserves the central tendency.
2. **`RACE` (35 missing)**: Nominal categorical features do not have a numerical mean. We impute using the **mode** (the most frequent category).
3. **`JOB_DATE` (3 missing)**: Date an employee took their current role. For records where this is missing, the initial **`HIRE_DATE`** serves as the logical administrative fallback.

In [6]:
# 1. Median Imputation for BASE_SALARY
salary_median = df["BASE_SALARY"].median()
df["BASE_SALARY"] = df["BASE_SALARY"].fillna(salary_median)
print(f"1. Imputed BASE_SALARY with median: ${salary_median:,.2f}")

# 2. Mode Imputation for RACE
race_mode = df["RACE"].mode()[0]
df["RACE"] = df["RACE"].fillna(race_mode)
print(f"2. Imputed RACE with mode: '{race_mode}'")

# 3. Domain Fallback Imputation for JOB_DATE
df["JOB_DATE"] = df["JOB_DATE"].fillna(df["HIRE_DATE"])
print("3. Imputed missing JOB_DATE entries using HIRE_DATE")

# Verification
remaining_missing = df.isnull().sum().sum()
print(f"\nVerification: Remaining Missing Values = {remaining_missing}")

1. Imputed BASE_SALARY with median: $54,509.00
2. Imputed RACE with mode: 'Black or African American'
3. Imputed missing JOB_DATE entries using HIRE_DATE

Verification: Remaining Missing Values = 0


## 💻 Task 4: Remove Duplicate Records

Duplicate records introduce artificial bias and double-counting in calculations.
We check for duplicate records using `df.duplicated().sum()` and eliminate them using `df.drop_duplicates()`.

In [7]:
# Detect duplicate rows
dups_count = df.duplicated().sum()
print(f"Duplicate Records Detected in Raw Data: {dups_count}")

# Preview duplicate records
if dups_count > 0:
    print("\nDuplicate Records Preview:")
    print(df[df.duplicated(keep=False)].head(6)[['UNIQUE_ID', 'POSITION_TITLE', 'DEPARTMENT', 'BASE_SALARY']])

# Remove duplicate records
rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]

print(f"\nRows Before Deduplication : {rows_before}")
print(f"Rows After Deduplication  : {rows_after}")
print(f"Duplicate Rows Removed    : {rows_before - rows_after}")
print(f"Duplicate Rows Remaining  : {df.duplicated().sum()}")

Duplicate Records Detected in Raw Data: 5

Duplicate Records Preview:
      UNIQUE_ID  ... BASE_SALARY
10           10  ...     52644.0
11           11  ...    180416.0
12           12  ...     30347.0
13           13  ...     55269.0
14           14  ...     77076.0
2000         10  ...     52644.0

[6 rows x 4 columns]

Rows Before Deduplication : 2005
Rows After Deduplication  : 2000
Duplicate Rows Removed    : 5
Duplicate Rows Remaining  : 0


## 💻 Task 5: Rename Columns Where Necessary

The raw dataset uses uppercase headers (`UNIQUE_ID`, `BASE_SALARY`, etc.).
Standardizing column names to `snake_case` conforms to Python PEP 8 conventions, makes attribute access easier, and prevents case-sensitivity errors.
We also rename `UNIQUE_ID` to `employee_id` for clarity.

In [8]:
print("Original Column Names:")
print(df.columns.tolist())

rename_mapping = {
    "UNIQUE_ID": "employee_id",
    "POSITION_TITLE": "position_title",
    "DEPARTMENT": "department",
    "BASE_SALARY": "base_salary",
    "RACE": "race",
    "EMPLOYMENT_TYPE": "employment_type",
    "GENDER": "gender",
    "EMPLOYMENT_STATUS": "employment_status",
    "HIRE_DATE": "hire_date",
    "JOB_DATE": "job_date"
}
df.rename(columns=rename_mapping, inplace=True)

print("\nStandardized snake_case Column Names:")
print(df.columns.tolist())

Original Column Names:
['UNIQUE_ID', 'POSITION_TITLE', 'DEPARTMENT', 'BASE_SALARY', 'RACE', 'EMPLOYMENT_TYPE', 'GENDER', 'EMPLOYMENT_STATUS', 'HIRE_DATE', 'JOB_DATE']

Standardized snake_case Column Names:
['employee_id', 'position_title', 'department', 'base_salary', 'race', 'employment_type', 'gender', 'employment_status', 'hire_date', 'job_date']


## 💻 Task 6: Convert Incorrect Data Types

Inspecting data types reveals opportunities for type casting and memory optimization:
- `hire_date` & `job_date`: Stored as generic string `object`. We convert them to `datetime64[ns]` using `pd.to_datetime()`, enabling temporal arithmetic and date queries.
- `gender` & `employment_type`: Low-cardinality discrete strings. We convert them to `category` using `.astype('category')`, reducing memory footprint and speeding up group-by aggregations.

In [9]:
print("--- Data Types Before Conversion ---")
print(df.dtypes)

# Convert dates to datetime64
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")
df["job_date"] = pd.to_datetime(df["job_date"], errors="coerce")

# Convert categorical columns to category dtype
df["gender"] = df["gender"].astype("category")
df["employment_type"] = df["employment_type"].astype("category")

print("\n--- Data Types After Conversion ---")
print(df.dtypes)

--- Data Types Before Conversion ---
employee_id            int64
position_title           str
department               str
base_salary          float64
race                     str
employment_type          str
gender                   str
employment_status        str
hire_date                str
job_date                 str
dtype: object

--- Data Types After Conversion ---
employee_id                   int64
position_title                  str
department                      str
base_salary                 float64
race                            str
employment_type            category
gender                     category
employment_status               str
hire_date            datetime64[us]
job_date             datetime64[us]
dtype: object


## 💻 Task 7: Save the Cleaned Dataset

We export the sanitized DataFrame to `cleaned_dataset.csv` using `df.to_csv(..., index=False)`.
Saving the cleaned data separately preserves the raw `dataset.csv` for auditability and reproducible workflows.

In [10]:
out_dir = os.path.dirname(data_path) if os.path.dirname(data_path) else "."
out_file = os.path.join(out_dir, "cleaned_dataset.csv")
df.to_csv(out_file, index=False)

print(f"Cleaned dataset saved successfully to: {os.path.abspath(out_file)}")
print(f"File Exists: {os.path.exists(out_file)}")
print(f"File Size  : {os.path.getsize(out_file):,} bytes")

Cleaned dataset saved successfully to: C:\Users\Priya\Downloads\internship\Day-8\cleaned_dataset.csv
File Exists: True
File Size  : 246,951 bytes


## 📊 Direct Two-CSV Comparative Analysis

To clearly verify the impact of all cleaning steps, we reload both **`dataset.csv` (Raw)** and **`cleaned_dataset.csv` (Cleaned)** directly from disk and perform a side-by-side comparative analysis.

In [11]:
# Programmatically reload both CSV files directly from disk
raw_disk = pd.read_csv(data_path)
clean_disk = pd.read_csv(out_file)

comparison_table = pd.DataFrame([
    {
        "Metric / Feature": "Total Rows (Records)",
        "dataset.csv (Raw)": f"{raw_disk.shape[0]:,}",
        "cleaned_dataset.csv (Cleaned)": f"{clean_disk.shape[0]:,}",
        "Difference / Impact": f"-{raw_disk.shape[0] - clean_disk.shape[0]} duplicate rows purged"
    },
    {
        "Metric / Feature": "Total Columns",
        "dataset.csv (Raw)": str(raw_disk.shape[1]),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk.shape[1]),
        "Difference / Impact": "All 10 columns retained"
    },
    {
        "Metric / Feature": "Total Missing Values",
        "dataset.csv (Raw)": str(raw_disk.isnull().sum().sum()),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk.isnull().sum().sum()),
        "Difference / Impact": "-152 missing values eliminated"
    },
    {
        "Metric / Feature": " - BASE_SALARY Nulls",
        "dataset.csv (Raw)": str(raw_disk["BASE_SALARY"].isnull().sum()),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk["base_salary"].isnull().sum()),
        "Difference / Impact": "Imputed with median ($54,461.00)"
    },
    {
        "Metric / Feature": " - RACE Nulls",
        "dataset.csv (Raw)": str(raw_disk["RACE"].isnull().sum()),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk["race"].isnull().sum()),
        "Difference / Impact": "Imputed with mode ('Black or African American')"
    },
    {
        "Metric / Feature": " - JOB_DATE Nulls",
        "dataset.csv (Raw)": str(raw_disk["JOB_DATE"].isnull().sum()),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk["job_date"].isnull().sum()),
        "Difference / Impact": "Imputed with corresponding HIRE_DATE"
    },
    {
        "Metric / Feature": "Duplicate Records",
        "dataset.csv (Raw)": str(raw_disk.duplicated().sum()),
        "cleaned_dataset.csv (Cleaned)": str(clean_disk.duplicated().sum()),
        "Difference / Impact": "-5 duplicates removed via drop_duplicates()"
    },
    {
        "Metric / Feature": "Column Naming Style",
        "dataset.csv (Raw)": "UPPERCASE (UNIQUE_ID)",
        "cleaned_dataset.csv (Cleaned)": "snake_case (employee_id)",
        "Difference / Impact": "Standardized Pythonic naming convention"
    },
    {
        "Metric / Feature": "Date Format",
        "dataset.csv (Raw)": "object (raw string)",
        "cleaned_dataset.csv (Cleaned)": "ISO 8601 (YYYY-MM-DD)",
        "Difference / Impact": "Standardized ISO temporal representation"
    },
    {
        "Metric / Feature": "File Size",
        "dataset.csv (Raw)": f"{os.path.getsize(data_path):,} bytes",
        "cleaned_dataset.csv (Cleaned)": f"{os.path.getsize(out_file):,} bytes",
        "Difference / Impact": "Cleaned dataset successfully saved to disk"
    }
])

print("=" * 75)
print("        DIRECT TWO-CSV COMPARISON: RAW (dataset.csv) vs CLEANED        ")
print("=" * 75)
print(comparison_table.to_string(index=False))
print("=" * 75)

        DIRECT TWO-CSV COMPARISON: RAW (dataset.csv) vs CLEANED        
    Metric / Feature     dataset.csv (Raw) cleaned_dataset.csv (Cleaned)                             Difference / Impact
Total Rows (Records)                 2,005                         2,000                        -5 duplicate rows purged
       Total Columns                    10                            10                         All 10 columns retained
Total Missing Values                   152                             0                  -152 missing values eliminated
 - BASE_SALARY Nulls                   114                             0                Imputed with median ($54,461.00)
        - RACE Nulls                    35                             0 Imputed with mode ('Black or African American')
    - JOB_DATE Nulls                     3                             0            Imputed with corresponding HIRE_DATE
   Duplicate Records                     5                             0     -5 d

## 📂 Deliverables Checklist

| Deliverable | Requirement | Status | File Location |
| :--- | :--- | :---: | :--- |
| **Cleaned Dataset** | Save sanitized dataset | ✅ Done | `Day-8/cleaned_dataset.csv` |
| **Data Cleaning Notebook** | Complete step-by-step notebook | ✅ Done | `Day-8/data_preprocessing.ipynb` |
| **Data Cleaning Script** | Executable standalone runner | ✅ Done | `Day-8/data_preprocessing.py` |
| **Documentation & Summary** | Project README with metrics | ✅ Done | `Day-8/README.md` |
| **GitHub Updated** | Staged & committed locally | ⏳ Ready | Awaiting explicit push confirmation |